In [16]:
import pandas as pd

In [17]:
tsv = pd.read_csv("./all.tsv", sep="\t")
# drop Protein families na values
tsv = tsv.dropna(subset=["Protein families"])
tsv.head()

,Entry,Protein names,Protein families,Keyword ID,Keywords,Organism,Sequence
0,A0A026W182,Odorant receptor coreceptor,"Insect chemoreceptor superfamily, Heteromeric ...",KW-0085; KW-1003; KW-0325; KW-0472; KW-0552; K...,Behavior;Cell membrane;Glycoprotein;Membrane;O...,Ooceraea biroi (Clonal raider ant) (Cerapachys...,MMKMKQQGLVADLLPNIRVMKTFGHFVFNYYNDNSSKYLHKVYCCV...
1,A0A044RE18,Endoprotease bli (EC 3.4.21.75) (Blisterase),"Peptidase S8 family, Furin subfamily",KW-0068; KW-0106; KW-0165; KW-1015; KW-0325; K...,Autocatalytic cleavage;Calcium;Cleavage on pai...,Onchocerca volvulus,MYWQLVRILVLFDCLQKILAIEHDSICIADVDDACPEPSHTVMRLR...
2,A0A061ACU2,Piezo-type mechanosensitive ion channel compon...,PIEZO (TC 1.A.75) family,KW-0002; KW-0024; KW-0025; KW-1003; KW-0325; K...,3D-structure;Alternative initiation;Alternativ...,Caenorhabditis elegans,MTVPPLLKSCVVKLLLPAALLAAAIIRPSFLSIGYVLLALVSAVLP...
3,A0A061I403,Protein adenylyltransferase FICD (EC 2.7.7.108...,Fic family,KW-0067; KW-0256; KW-0325; KW-0378; KW-0460; K...,ATP-binding;Endoplasmic reticulum;Glycoprotein...,Cricetulus griseus (Chinese hamster) (Cricetul...,MPMASVIAVAEPKWISVWGRFLWLTLLSMALGSLLALLLPLGAVEE...
4,A0A075F932,Synaptotagmin-1 (Synaptotagmin I) (SytI),Synaptotagmin family,KW-0106; KW-0963; KW-0968; KW-0221; KW-0325; K...,Calcium;Cytoplasm;Cytoplasmic vesicle;Differen...,Anser cygnoides (Swan goose),MVSESHHEALAAPPATTVAAAPPSNVTEPASPGGGGGKEDAFSKLK...


In [18]:
# if Keyword_ID contains KW-0800, add a new column called "Toxin" and set it to True, else set it to False
tsv["Toxin"] = tsv["Keyword ID"].str.contains("KW-0800")
tsv["Toxin"].value_counts()

Toxin
False    79363
True      5567
Name: count, dtype: int64

In [19]:
# tsv["Protein families"].value_counts()

# apply the same family/superfamily normalization as in preprocessing.load_and_prepare_raw
# first, normalize the raw "Protein families" column
tsv["Protein families"] = tsv["Protein families"].str.split(";").str[0]
tsv["Protein families"] = tsv["Protein families"].str.split(",").str[0]

repl = {
    "I1 superfamily": "Conotoxin I1 superfamily",
    "O1 superfamily": "Conotoxin O1 superfamily",
    "O2 superfamily": "Conotoxin O2 superfamily",
    "E superfamily": "Conotoxin E superfamily",
    "F superfamily": "Conotoxin F superfamily",
}
tsv["Protein families"] = tsv["Protein families"].replace(repl)

mapping = {
    r"Conotoxin.*": "Conotoxin family",
    r"Neurotoxin.*": "Neurotoxin family",
    r"Scoloptoxin.*|Scolopendra.*": "Scoloptoxin family",
    r"Caterpillar.*": "Caterpillar family",
    r"Teretoxin.*": "Teretoxin family",
    r"Limacoditoxin.*": "Limacoditoxin family",
    r"Scutigerotoxin.*": "Scutigerotoxin family",
    r"Cationic peptide.*": "Cationic peptide family",
    r"Formicidae venom.*": "Formicidae venom family",
    r"Sea anemone.*potassium channel toxin family.*": "Sea anemone potassium channel toxin family",
    r"Bradykinin-potentiating peptide family|Natriuretic peptide family|Natriuretic": "Natriuretic, Bradykinin potentiating peptide family",
    r".*phospholipase.*|.*Phospholipase.*": "Phospholipase family",
    r"Peptidase.*": "Peptidase family",
    r"FARP.*": "FARP family"
}

for pattern, replacement in mapping.items():
    tsv["Protein families"] = tsv["Protein families"].str.replace(
        pattern, replacement, regex=True
    )

# rename all that have 10 or less than 10 samples in "Protein families" into "Other"
vc = tsv["Protein families"].value_counts()

tsv["Protein families"] = tsv["Protein families"].where(
    tsv["Protein families"].map(vc) > 20,
    other="Other"
)

print(len(tsv["Protein families"].unique()))

# save all unique values in the Protein family column in a tsv file for Toxin column values that are True and False each
tsv[tsv["Toxin"] == True]["Protein families"].value_counts().to_csv("/Users/selin/Desktop/Toxin_Pfams.csv")
tsv[tsv["Toxin"] == False]["Protein families"].value_counts().to_csv("/Users/selin/Desktop/Non_Toxin_Pfams.csv")
tsv["Protein families"].value_counts().to_csv("/Users/selin/Desktop/Pfams.csv")

684


In [20]:
import difflib

# unique families for toxin / non-toxin (after your normalization)
tox_fams = sorted(tsv.loc[tsv["Toxin"] == True, "Protein families"].dropna().unique())
nontox_fams = sorted(tsv.loc[tsv["Toxin"] == False, "Protein families"].dropna().unique())

# precompute counts for each family in toxin / non-toxin parts
tox_counts = tsv.loc[tsv["Toxin"] == True, "Protein families"].value_counts()
nontox_counts = tsv.loc[tsv["Toxin"] == False, "Protein families"].value_counts()

def best_match(name, candidates):
    best, best_score = None, 0.0
    for cand in candidates:
        score = difflib.SequenceMatcher(None, name, cand).ratio()
        if score > best_score:
            best, best_score = cand, score
    return best, best_score

rows = []
threshold = 0.88  # tune this (0–1) for stricter/looser matching

for fam in nontox_fams:
    match, score = best_match(fam, tox_fams)
    if score >= threshold:
        rows.append(
            (
                fam,
                match,
                score,
                nontox_counts.get(fam, 0),
                tox_counts.get(match, 0),
            )
        )

similar_non_tox = pd.DataFrame(
    rows,
    columns=[
        "Non-toxin family",
        "Closest toxin family",
        "similarity",
        "Non-toxin count",
        "Toxin count",
    ],
).sort_values("similarity", ascending=False)


similar_non_tox

,Non-toxin family,Closest toxin family,similarity,Non-toxin count,Toxin count
0,AB hydrolase superfamily,AB hydrolase superfamily,1.000000,258,2
25,Multicopper oxidase family,Multicopper oxidase family,1.000000,19,4
27,NPY family,NPY family,1.000000,104,4
28,"Natriuretic, Bradykinin potentiating peptide f...","Natriuretic, Bradykinin potentiating peptide f...",1.000000,68,100
29,Neurotoxin family,Neurotoxin family,1.000000,9,962
30,Non-disulfide-bridged peptide (NDBP) superfamily,Non-disulfide-bridged peptide (NDBP) superfamily,1.000000,138,33
31,Nucleotide pyrophosphatase/phosphodiesterase f...,Nucleotide pyrophosphatase/phosphodiesterase f...,1.000000,32,5
32,Other,Other,1.000000,27649,381
33,PBP/GOBP family,PBP/GOBP family,1.000000,76,16
34,PDGF/VEGF growth factor family,PDGF/VEGF growth factor family,1.000000,51,18


In [21]:
tsv

,Entry,Protein names,Protein families,Keyword ID,Keywords,Organism,Sequence,Toxin
0,A0A026W182,Odorant receptor coreceptor,Insect chemoreceptor superfamily,KW-0085; KW-1003; KW-0325; KW-0472; KW-0552; K...,Behavior;Cell membrane;Glycoprotein;Membrane;O...,Ooceraea biroi (Clonal raider ant) (Cerapachys...,MMKMKQQGLVADLLPNIRVMKTFGHFVFNYYNDNSSKYLHKVYCCV...,False
1,A0A044RE18,Endoprotease bli (EC 3.4.21.75) (Blisterase),Peptidase family,KW-0068; KW-0106; KW-0165; KW-1015; KW-0325; K...,Autocatalytic cleavage;Calcium;Cleavage on pai...,Onchocerca volvulus,MYWQLVRILVLFDCLQKILAIEHDSICIADVDDACPEPSHTVMRLR...,False
2,A0A061ACU2,Piezo-type mechanosensitive ion channel compon...,Other,KW-0002; KW-0024; KW-0025; KW-1003; KW-0325; K...,3D-structure;Alternative initiation;Alternativ...,Caenorhabditis elegans,MTVPPLLKSCVVKLLLPAALLAAAIIRPSFLSIGYVLLALVSAVLP...,False
3,A0A061I403,Protein adenylyltransferase FICD (EC 2.7.7.108...,Fic family,KW-0067; KW-0256; KW-0325; KW-0378; KW-0460; K...,ATP-binding;Endoplasmic reticulum;Glycoprotein...,Cricetulus griseus (Chinese hamster) (Cricetul...,MPMASVIAVAEPKWISVWGRFLWLTLLSMALGSLLALLLPLGAVEE...,False
4,A0A075F932,Synaptotagmin-1 (Synaptotagmin I) (SytI),Synaptotagmin family,KW-0106; KW-0963; KW-0968; KW-0221; KW-0325; K...,Calcium;Cytoplasm;Cytoplasmic vesicle;Differen...,Anser cygnoides (Swan goose),MVSESHHEALAAPPATTVAAAPPSNVTEPASPGGGGGKEDAFSKLK...,False
...,...,...,...,...,...,...,...,...
105756,Q9W3M2,DM7 family protein CG15332,Other,KW-1185; KW-0677,Reference proteome;Repeat,Drosophila melanogaster (Fruit fly),MAKRGKKGGIPRAEMVQVASANRDENQVTELKKADYLPYLFNLVMP...,False
105757,Q9WUQ7,Dexamethasone-induced protein (Protein MYLE),Other,KW-1185,Reference proteome,Mus musculus (Mouse),MPGARVAAHLDALGPLVSYVQPPLLPSMFYVGLFFVNVLILYYAFL...,False
105760,Q9XS96,SNRPN upstream reading frame protein,Other,KW-0539; KW-1185,Nucleus;Reference proteome,Bos taurus (Bovine),MERARDRLHLRRTTEQHVPEVEVQVKRRRTASLNNQECHVYLRRSQ...,False
105761,Q9XS97,SNRPN upstream reading frame protein,Other,KW-0539; KW-1185,Nucleus;Reference proteome,Oryctolagus cuniculus (Rabbit),MERARDRLHLRRTTEQHVPEVEVQVKRRRTASLSNQECQLYPRRSQ...,False
